# Level 2 — Task 3: K-Means Clustering
**Dataset:** Churn (bigml-20 + bigml-80 merged) | **Tools:** pandas, scikit-learn, matplotlib, seaborn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")


## 2. Load & Merge Dataset

In [ ]:
from google.colab import files
print("Upload both churn-bigml-80.csv and churn-bigml-20.csv")
uploaded = files.upload()

dfs = [pd.read_csv(f) for f in uploaded.keys()]
df = pd.concat(dfs, ignore_index=True)

print(f"Merged shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


## 3. Exploratory Data Analysis

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["Churn"].value_counts().plot(kind="bar", ax=axes[0], color=["#4C72B0","#DD8452"], edgecolor="white")
axes[0].set_title("Churn Distribution")
axes[0].set_xticklabels(["False","True"], rotation=0)

df["International plan"].value_counts().plot(kind="bar", ax=axes[1], color=["#55A868","#C44E52"], edgecolor="white")
axes[1].set_title("International Plan Distribution")
axes[1].set_xticklabels(["No","Yes"], rotation=0)

plt.tight_layout()
plt.show()


## 4. Preprocessing

In [ ]:
df_proc = df.copy()

# Drop non-informative columns
df_proc.drop(columns=["State", "Area code"], inplace=True)

# Encode binary categoricals
binary_cols = ["International plan", "Voice mail plan", "Churn"]
le = LabelEncoder()
for col in binary_cols:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

# Confirm no missing values
assert df_proc.isnull().sum().sum() == 0, "Unexpected missing values"
print("No missing values. Shape:", df_proc.shape)
df_proc.head()


In [ ]:
# Scale features — K-Means is distance-based, scaling is critical
feature_cols = df_proc.columns.tolist()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_proc[feature_cols])

print(f"Feature matrix: {X_scaled.shape}")


## 5. Elbow Method — Find Optimal K

In [ ]:
inertia = []
silhouette = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertia, marker="o", color="#4C72B0")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_title("Elbow Method")

axes[1].plot(k_range, silhouette, marker="s", color="#DD8452")
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score vs K")

plt.suptitle("Optimal K Selection", fontweight="bold")
plt.tight_layout()
plt.show()

optimal_k = k_range[np.argmax(silhouette)]
print(f"Optimal K by silhouette score: {optimal_k}")


## 6. Train Final K-Means Model

In [ ]:
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
km_final.fit(X_scaled)

df_proc["Cluster"] = km_final.labels_

print(f"Cluster distribution:")
print(df_proc["Cluster"].value_counts().sort_index())
print(f"\nInertia       : {km_final.inertia_:.2f}")
print(f"Silhouette    : {silhouette_score(X_scaled, km_final.labels_):.4f}")


## 7. Visualize Clusters (PCA 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"Variance explained by PC1: {explained[0]:.2%}")
print(f"Variance explained by PC2: {explained[1]:.2%}")
print(f"Total                    : {explained.sum():.2%}")

df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df_pca["Cluster"] = km_final.labels_
df_pca["Churn"] = df_proc["Churn"].values

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Cluster view
palette = sns.color_palette("tab10", optimal_k)
for cluster_id in range(optimal_k):
    mask = df_pca["Cluster"] == cluster_id
    axes[0].scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC2"],
                    s=15, alpha=0.6, label=f"Cluster {cluster_id}", color=palette[cluster_id])

centroids_pca = pca.transform(km_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                marker="X", s=200, color="black", zorder=5, label="Centroids")
axes[0].set_title(f"K-Means Clusters (K={optimal_k}) — PCA Projection")
axes[0].set_xlabel(f"PC1 ({explained[0]:.1%} variance)")
axes[0].set_ylabel(f"PC2 ({explained[1]:.1%} variance)")
axes[0].legend(fontsize=8)

# Churn view
for churn_val, label, color in [(0, "Not Churned", "#4C72B0"), (1, "Churned", "#DD8452")]:
    mask = df_pca["Churn"] == churn_val
    axes[1].scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC2"],
                    s=15, alpha=0.5, label=label, color=color)

axes[1].set_title("Actual Churn Labels — PCA Projection")
axes[1].set_xlabel(f"PC1 ({explained[0]:.1%} variance)")
axes[1].set_ylabel(f"PC2 ({explained[1]:.1%} variance)")
axes[1].legend()

plt.suptitle("Cluster vs Actual Churn — PCA 2D Projection", fontweight="bold")
plt.tight_layout()
plt.show()


## 8. Cluster Profiling

In [ ]:
profile = df_proc.groupby("Cluster").mean().round(3)
print("Cluster Profiles (mean per feature):")
profile


In [ ]:
# Churn rate per cluster
churn_rate = df_proc.groupby("Cluster")["Churn"].mean().rename("Churn Rate")
cluster_size = df_proc["Cluster"].value_counts().sort_index().rename("Size")
summary = pd.concat([cluster_size, churn_rate], axis=1)
print("Cluster Summary:")
print(summary)

plt.figure(figsize=(7, 4))
summary["Churn Rate"].plot(kind="bar", color="#DD8452", edgecolor="white")
plt.xlabel("Cluster")
plt.ylabel("Churn Rate")
plt.title("Churn Rate per Cluster")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap of cluster profiles (normalized)
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())

plt.figure(figsize=(14, 4))
sns.heatmap(profile_norm, annot=True, fmt=".2f", cmap="YlOrRd", linewidths=0.4)
plt.title("Cluster Profile Heatmap (Min-Max Normalized)")
plt.tight_layout()
plt.show()


## 9. Summary

| Metric | Value |
|---|---|
| Optimal K | See output above |
| Inertia (WCSS) | See output above |
| Silhouette Score | See output above |
| PCA Variance Explained | See output above |

Clusters with high churn rates represent high-risk customer segments. The cluster profile heatmap shows which behavioral features (e.g., customer service calls, total day minutes) drive separation between groups.